# 04 - Portfolio Tests

This notebook tests SGI-aware portfolio strategies.

## Steps:
1. Run baseline portfolios
2. Run SGI-conditioned portfolios
3. Compare performance metrics
4. Analyze crisis behavior

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config import load_experiment_config
from src.experiments import run_signal_pipeline
from src.portfolio import generate_weight_schedule
from src.backtest import run_rebalancing_backtest, compare_strategies, compute_crisis_performance
from src.plots import plot_portfolio_comparison
from src.constants import CRISIS_DATES
from src.paths import ensure_all_directories

ensure_all_directories()

## Run Signal Pipeline

In [ ]:
config = load_experiment_config('../configs/universe_sector_etfs.yaml')
signal_results = run_signal_pipeline(config)

prices = signal_results['prices']
returns = signal_results['returns']
cov_matrices = signal_results['cov_matrices']
geometry_df = signal_results['geometry_df']

## Generate Portfolio Weights

In [ ]:
strategies = ['equal_weight', 'inverse_vol', 'min_variance', 'sgi_conditioned_minvar']

weight_schedules = {}
for strategy in strategies:
    print(f"Generating weights for: {strategy}")
    weights = generate_weight_schedule(
        returns, cov_matrices, geometry_df,
        strategy, config, 'weekly'
    )
    weight_schedules[strategy] = weights
    print(f"  Shape: {weights.shape}")

## Run Backtests

In [ ]:
transaction_cost = 10  # basis points

backtest_results = {}
portfolio_returns = {}

for strategy, weights in weight_schedules.items():
    results = run_rebalancing_backtest(prices, weights, transaction_cost)
    backtest_results[strategy] = results
    portfolio_returns[strategy] = results['net_returns']

In [ ]:
# Compare strategies
comparison = compare_strategies(prices, weight_schedules, transaction_cost)
comparison[['strategy', 'cagr', 'volatility', 'sharpe', 'max_drawdown', 'turnover_annual']]

## Visualize Performance

In [ ]:
plot_portfolio_comparison(portfolio_returns)

## Crisis Period Analysis

In [ ]:
# Analyze performance during crises
for strategy, returns in portfolio_returns.items():
    print(f"\n{strategy}:")
    crisis_perf = compute_crisis_performance(returns, CRISIS_DATES)
    if len(crisis_perf) > 0:
        print(crisis_perf[['crisis', 'total_return', 'max_drawdown']].to_string(index=False))

## Key Findings

1. Compare Sharpe ratios across strategies
2. Check if SGI-conditioned strategies reduce max drawdown
3. Analyze turnover and transaction cost impact